# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [9]:
# ===== Colab: kết nối Google Drive và chuyển vào submission_2A202602367/code =====
from google.colab import drive
drive.mount("/content/drive")
import os, sys
os.chdir("/content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork/submission_2A202602367/code")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

Mounted at /content/drive
cwd: /content/drive/MyDrive/K4-Track4-Day1-Neuralnetwork/submission_2A202602367/code


In [10]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
# Colab: nếu chưa có repo, bỏ comment 2 dòng dưới, rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

# TODO: chọn device ("cuda" nếu có GPU, ngược lại "cpu"; trên Mac có thể "mps"); in torch.__version__ và tên GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
# TODO: os.makedirs cho f"{OUT_DIR}/figures" và f"{OUT_DIR}/results"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True); os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

Device: cuda | torch 2.11.0+cu130 | Tesla T4


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [11]:
# 1. Kiểm tra và chia dữ liệu nếu chưa có file .npz
processed_dir = f"{REPO_ROOT}/data/processed"
if not (os.path.exists(f"{processed_dir}/train.npz") and os.path.exists(f"{processed_dir}/eval.npz")):
    print("Đang chia dữ liệu từ metadata (scripts/split_data.py)...")
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

# 2. Nạp dữ liệu, tách validation 20% và đưa toàn bộ lên GPU T4
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# 3. Kiểm tra thống kê 10 cột số liên tục trên train (mean ≈ 0, std ≈ 1)
mean_check = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
std_check = data["X_tr"][:, :10].std(dim=0).cpu().numpy()

print("\n=== KIỂM TRA CHUẨN HOÁ PART 0 ===")
print("Mean 10 cột số (kỳ vọng ≈ 0):", np.round(mean_check, 3))
print("Std 10 cột số  (kỳ vọng ≈ 1):", np.round(std_check, 3))

assert np.allclose(mean_check, 0.0, atol=1e-2), "Lỗi: Mean chưa về 0!"
assert np.allclose(std_check, 1.0, atol=1e-2), "Lỗi: Std chưa về 1!"
print("-> Toàn bộ dữ liệu đã được chuẩn hoá chuẩn xác và sẵn sàng trên GPU!")


Dữ liệu sẵn sàng trên cuda: Train=371,847, Val=92,962, Eval=116,203
Đoán đa số (lớp 1) trên val: accuracy = 0.4876 (≈ 0.4876)

=== KIỂM TRA CHUẨN HOÁ PART 0 ===
Mean 10 cột số (kỳ vọng ≈ 0): [ 0. -0.  0. -0. -0. -0.  0. -0.  0. -0.]
Std 10 cột số  (kỳ vọng ≈ 1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
-> Toàn bộ dữ liệu đã được chuẩn hoá chuẩn xác và sẵn sàng trên GPU!


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [12]:
import torch.nn.functional as F
import numpy as np

# 1. Khởi tạo mô hình M-base (54 -> 256 -> 128 -> 7)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"1. Số tham số M-base: {n_params} (Kỳ vọng chuẩn: {EXPECTED_PARAMS[(256, 128)]})")
assert n_params == EXPECTED_PARAMS[(256, 128)], "Số tham số không khớp!"

# 2. Kiểm tra shape logits đầu ra
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model(dummy_x)
assert dummy_logits.shape == (8, 7), f"Shape sai: {dummy_logits.shape}"
print(f"2. Shape logits: {dummy_logits.shape} -> Chuẩn xác (B, 7)")

# 3. Phép thử 1: In Loss bước 0 trên val
model.eval()
with torch.no_grad():
    val_logits = model(data["X_val"][:4096])
    step0_loss = F.cross_entropy(val_logits, data["y_val"][:4096]).item()
print(f"3. Loss bước 0 đo được: {step0_loss:.4f} (Lý thuyết ln(7) ≈ {np.log(7):.4f})")
print("   -> Nhận xét: Lệch so với ln(7) do phương sai khởi tạo He và mất cân bằng lớp (Lớp 1 chiếm 48.8%).")

# 4. Phép thử 2: Quá khớp (Overfit) 20 mẫu
overfit_x = data["X_tr"][:20]
overfit_y = data["y_tr"][:20]
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt = torch.optim.Adam(overfit_model.parameters(), lr=0.01)

for step in range(250):
    overfit_model.train()
    opt.zero_grad()
    loss = F.cross_entropy(overfit_model(overfit_x), overfit_y)
    loss.backward()
    opt.step()

pred_20 = overfit_model(overfit_x).argmax(dim=1)
acc_20 = (pred_20 == overfit_y).float().mean().item()
print(f"4. Quá khớp 20 mẫu: Loss = {loss.item():.6f}, Accuracy = {acc_20 * 100:.1f}%")
assert acc_20 == 1.0, "Mô hình không overfit được 100% trên 20 mẫu!"

# 5. Kiểm tra gradient chảy tới mọi tầng
for name, param in overfit_model.named_parameters():
    assert param.grad is not None and (param.grad.abs().sum().item() > 0), f"Gradient không chảy qua {name}!"
print("5. Gradient chảy thông suốt qua tất cả các trọng số và bias!")


1. Số tham số M-base: 47879 (Kỳ vọng chuẩn: 47879)
2. Shape logits: torch.Size([8, 7]) -> Chuẩn xác (B, 7)
3. Loss bước 0 đo được: 2.0641 (Lý thuyết ln(7) ≈ 1.9459)
   -> Nhận xét: Lệch so với ln(7) do phương sai khởi tạo He và mất cân bằng lớp (Lớp 1 chiếm 48.8%).
4. Quá khớp 20 mẫu: Loss = 0.000012, Accuracy = 100.0%
5. Gradient chảy thông suốt qua tất cả các trọng số và bias!


**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [13]:
# ==============================================================================
# PART 2: QUÉT LR, CHẠY BASELINE 3 SEEDS VÀ ĐO ĐỘ NHIỄU
# ==============================================================================
import numpy as np, pandas as pd

# 1. Quét nhanh chọn lr tối ưu cho baseline
print("=== 1. QUÉT CHỌN LEARNING RATE TỐI ƯU ===")
lr_candidates = [0.01, 0.03, 0.05, 0.1]
best_lr, best_scan_f1 = 0.05, -1.0
scan_rows = []

for t_lr in lr_candidates:
    s_cfg = {**DEFAULT_CFG, "exp_id": f"scan-lr-{t_lr}", "lr": t_lr, "epochs": 6, "seed": 42}
    s_res = run_experiment(s_cfg, data)
    f1 = s_res["summary"]["val_macro_f1"]
    scan_rows.append({"Learning Rate": t_lr, "Val Macro-F1 (6 ep)": round(f1, 4)})
    if f1 > best_scan_f1:
        best_scan_f1, best_lr = f1, t_lr

print(pd.DataFrame(scan_rows).to_string(index=False))
print(f"-> Chọn được lr tối ưu cho Baseline: lr = {best_lr}\n")

# 2. Huấn luyện Baseline 20 epochs trên 3 seeds
print("=== 2. CHẠY BASELINE 3 SEEDS (20 EPOCHS) ===")
base_cfg = {**DEFAULT_CFG, "lr": best_lr, "epochs": 20}
baseline_results = []

for s in [1, 2, 3]:
    exp_id = f"base-s{s}"
    cfg_s = {**base_cfg, "seed": s, "exp_id": exp_id}
    res = run_experiment(cfg_s, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_id}.png")
    baseline_results.append(res)
    sm = res["summary"]
    print(f"[{exp_id}] Best Ep: {sm['best_epoch']} | Val Loss: {sm['best_val_loss']:.4f} | Val F1: {sm['val_macro_f1']:.4f}")

b_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
b_accs = [r["summary"]["val_acc"] for r in baseline_results]
mean_f1, std_f1 = np.mean(b_f1s), np.std(b_f1s, ddof=1)
mean_acc, std_acc = np.mean(b_accs), np.std(b_accs, ddof=1)
noise_2sigma = 2 * std_f1

print("\n" + "="*50)
print(f"Baseline Val Acc     : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Baseline Val Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Ngưỡng nhiễu 2σ (F1) : {noise_2sigma:.4f}")
print("="*50)


=== 1. QUÉT CHỌN LEARNING RATE TỐI ƯU ===
 Learning Rate  Val Macro-F1 (6 ep)
          0.01               0.6600
          0.03               0.7157
          0.05               0.7501
          0.10               0.7886
-> Chọn được lr tối ưu cho Baseline: lr = 0.1

=== 2. CHẠY BASELINE 3 SEEDS (20 EPOCHS) ===
[base-s1] Best Ep: 18 | Val Loss: 0.2393 | Val F1: 0.8434
[base-s2] Best Ep: 20 | Val Loss: 0.2246 | Val F1: 0.8654
[base-s3] Best Ep: 20 | Val Loss: 0.2280 | Val F1: 0.8505

Baseline Val Acc     : 0.9078 ± 0.0038
Baseline Val Macro-F1: 0.8531 ± 0.0112
Ngưỡng nhiễu 2σ (F1) : 0.0224


**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [14]:
# ==============================================================================
# PART 3: CHẠY CÁC THÍ NGHIỆM THEO 7 CHỦ ĐỀ & VẼ BIỂU ĐỒ
# ==============================================================================
experiments_to_run = [
    # Chủ đề 1: Loss
    {**base_cfg, "exp_id": "loss-mse", "group": "loss", "description": "Loss MSE", "loss": "mse"},
    # Chủ đề 2: Optimizer
    {**base_cfg, "exp_id": "opt-adam-lr1e-3", "group": "optimizer", "description": "Adam lr=1e-3", "optimizer": "adam", "lr": 1e-3},
    {**base_cfg, "exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "description": "AdamW lr=1e-3 wd=0.01", "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.01},
    {**base_cfg, "exp_id": "opt-sgd-lr0.05", "group": "optimizer", "description": "SGD thuần (không momentum)", "optimizer": "sgd", "lr": 0.05},
    # Chủ đề 3: Hyper-parameters
    {**base_cfg, "exp_id": "hparam-bs128", "group": "hparam", "description": "Batch 128", "batch": 128},
    {**base_cfg, "exp_id": "hparam-bs2048", "group": "hparam", "description": "Batch 2048", "batch": 2048},
    {**base_cfg, "exp_id": "hparam-wide", "group": "hparam", "description": "M-wide (512-256)", "hidden": (512, 256)},
    # Chủ đề 4: Dropout
    {**base_cfg, "exp_id": "drop-0.2", "group": "dropout", "description": "Dropout q=0.2", "dropout": 0.2},
    # Chủ đề 5: Gradient Clipping
    {**base_cfg, "exp_id": "clip-1.0", "group": "clipping", "description": "Clip norm c=1.0", "clip_norm": 1.0},
    {**base_cfg, "exp_id": "clip-highlr-clip1.0", "group": "clipping", "description": "lr cao (0.5) CÓ clip c=1.0", "lr": 0.5, "clip_norm": 1.0},
    # Chủ đề 6: Mixed Precision (AMP)
    {**base_cfg, "exp_id": "amp-fp16", "group": "amp", "description": "FP16 AMP", "precision": "fp16" if device == "cuda" else "fp32"},
    # Chủ đề 7: Khởi tạo tham số
    {**base_cfg, "exp_id": "init-xavier", "group": "init", "description": "Khởi tạo Xavier", "init": "xavier"},
    {**base_cfg, "exp_id": "init-zeros", "group": "init", "description": "Khởi tạo toàn 0", "init": "zeros"},
]

print(f"=== ĐANG CHẠY {len(experiments_to_run)} THÍ NGHIỆM ===")
experiment_results = []
for cfg_exp in experiments_to_run:
    eid = cfg_exp["exp_id"]
    print(f"-> [{eid}]...", end=" ")
    res = run_experiment(cfg_exp, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{eid}.png")
    experiment_results.append(res)
    sm = res["summary"]
    diff = sm["val_macro_f1"] - mean_f1
    print(f"Xong! Val F1: {sm['val_macro_f1']:.4f} (Lệch vs Base: {diff:+.4f})")

# Vẽ ảnh so sánh theo nhóm
print("\n=== ĐANG VẼ CÁC BIỂU ĐỒ SO SÁNH NHÓM ===")
groups = sorted(list(set(r["cfg"]["group"] for r in experiment_results)))
for g in groups:
    group_runs = [baseline_results[0]] + [r for r in experiment_results if r["cfg"]["group"] == g]
    plot_compare(group_runs, "val_macro_f1", f"{OUT_DIR}/figures/compare_{g}.png", title=f"So sánh Val Macro-F1 nhóm {g.upper()}")
print("✅ Đã lưu toàn bộ biểu đồ vào figures/!")


=== ĐANG CHẠY 13 THÍ NGHIỆM ===
-> [loss-mse]... Xong! Val F1: 0.7278 (Lệch vs Base: -0.1254)
-> [opt-adam-lr1e-3]... Xong! Val F1: 0.8446 (Lệch vs Base: -0.0086)
-> [opt-adamw-lr1e-3]... Xong! Val F1: 0.8403 (Lệch vs Base: -0.0129)
-> [opt-sgd-lr0.05]... Xong! Val F1: 0.6865 (Lệch vs Base: -0.1666)
-> [hparam-bs128]... Xong! Val F1: 0.8613 (Lệch vs Base: +0.0081)
-> [hparam-bs2048]... Xong! Val F1: 0.7992 (Lệch vs Base: -0.0540)
-> [hparam-wide]... Xong! Val F1: 0.8675 (Lệch vs Base: +0.0144)
-> [drop-0.2]... Xong! Val F1: 0.8230 (Lệch vs Base: -0.0301)
-> [clip-1.0]... Xong! Val F1: 0.8444 (Lệch vs Base: -0.0087)
-> [clip-highlr-clip1.0]... Xong! Val F1: 0.8518 (Lệch vs Base: -0.0013)
-> [amp-fp16]... Xong! Val F1: 0.8429 (Lệch vs Base: -0.0102)
-> [init-xavier]... Xong! Val F1: 0.8498 (Lệch vs Base: -0.0034)
-> [init-zeros]... Xong! Val F1: 0.0936 (Lệch vs Base: -0.7595)

=== ĐANG VẼ CÁC BIỂU ĐỒ SO SÁNH NHÓM ===
✅ Đã lưu toàn bộ biểu đồ vào figures/!


**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [15]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [16]:
# ==============================================================================
# PART 4: ĐÁNH GIÁ TRÊN EVAL & XUẤT BẢNG EXCEL
# ==============================================================================
import os, sys, subprocess, json, openpyxl
import pandas as pd, numpy as np

# 1. Chọn cấu hình có Val Macro-F1 cao nhất
all_runs = baseline_results + experiment_results
best_run = max(all_runs, key=lambda r: r["summary"]["val_macro_f1"])
best_cfg = best_run["cfg"]
print(f"CẤU HÌNH TỐT NHẤT: {best_cfg['exp_id']} (Val F1: {best_run['summary']['val_macro_f1']:.4f})\n")

# 2. Dự đoán tập eval cho cấu hình tốt nhất
pred_eval_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval(best_cfg, best_run, data, pred_eval_path)

eval_json_path = f"{OUT_DIR}/eval_result.json"
cmd = [sys.executable, "scripts/evaluate.py", "--pred", os.path.abspath(pred_eval_path), "--out", os.path.abspath(eval_json_path)]
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

# 3. Dự đoán tập eval cho baseline
base_pred_path = f"{OUT_DIR}/predictions_base.csv"
final_eval(baseline_results[0]["cfg"], baseline_results[0], data, base_pred_path)
base_json_path = f"{OUT_DIR}/eval_base_result.json"
subprocess.run([sys.executable, "scripts/evaluate.py", "--pred", os.path.abspath(base_pred_path), "--out", os.path.abspath(base_json_path)], cwd=REPO_ROOT, check=True)

with open(eval_json_path, "r", encoding="utf-8") as f:
    eval_res = json.load(f)
with open(base_json_path, "r", encoding="utf-8") as f:
    base_eval_res = json.load(f)

print("="*60)
print(f"BASELINE EVAL MACRO-F1: {base_eval_res['macro_f1']:.4f}")
print(f"BEST EXP EVAL MACRO-F1: {eval_res['macro_f1']:.4f}")
print(f"CẢI THIỆN             : {eval_res['macro_f1'] - base_eval_res['macro_f1']:+.4f}")
print("="*60 + "\n")

# 4. Xuất file experiments.xlsx
results_all = load_results(f"{OUT_DIR}/results")
rows = []
for r in results_all:
    eid = r["cfg"]["exp_id"]
    if eid == best_cfg["exp_id"]:
        rows.append(to_row(r, eval_scores=eval_res, notes="Cấu hình nộp bài"))
    elif eid == "base-s1":
        rows.append(to_row(r, eval_scores=base_eval_res, notes="Baseline seed 1"))
    else:
        rows.append(to_row(r, eval_scores=None))

template_path = f"{REPO_ROOT}/templates/experiment_table_template.xlsx"
out_excel_path = f"{OUT_DIR}/experiments.xlsx"
write_xlsx(rows, template_path, out_excel_path)

# Cập nhật sheet Seeds
wb = openpyxl.load_workbook(out_excel_path)
if "Seeds" in wb.sheetnames:
    ws_seeds = wb["Seeds"]
    for idx, s in enumerate([1, 2, 3], start=2):
        ws_seeds.cell(row=idx, column=1, value=f"base-s{s}")
wb.save(out_excel_path)
print("✅ ĐÃ XUẤT THÀNH CÔNG EXPERIMENTS.XLSX!")


CẤU HÌNH TỐT NHẤT: hparam-wide (Val F1: 0.8675)

Đã lưu dự đoán (116,203 dòng) vào: ../predictions_eval.csv
Đã lưu dự đoán (116,203 dòng) vào: ../predictions_base.csv
BASELINE EVAL MACRO-F1: 0.8429
BEST EXP EVAL MACRO-F1: 0.8696
CẢI THIỆN             : +0.0267

Đã lưu bảng Excel thành công vào: ../experiments.xlsx
✅ ĐÃ XUẤT THÀNH CÔNG EXPERIMENTS.XLSX!


In [17]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [18]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary